# Distill a **non-reasoning** 8B teacher -> Qwen3-0.6B — 3-seed control (disconnect-safe)

Non-reasoning-teacher control for the distillation study. Identical to the main
run in every respect (recipe, LoRA rank, hyperparameters, seeds 42/123/7,
q4_k_m export) EXCEPT the training data, which was re-labeled by
**Llama-3.1-8B-Instruct** instead of DeepSeek-R1-8B.

### Protected against disconnects
Colab runtimes are ephemeral: anything on the runtime disk is lost on
disconnect. So this notebook writes every artifact to **Google Drive the moment
a seed finishes**, and **resumes** on rerun (skips seeds already in Drive). A
disconnect therefore costs at most the one seed in progress -- never a finished
one, and never a download race.

**Steps:** (1) Runtime -> Change runtime type -> **T4 GPU**. (2) Run the Drive-
mount cell and approve the popup. (3) Upload `train_chat_llama_final.jsonl` via
the Files panel. (4) Run all cells. If you get disconnected, just reconnect and
**Run all again** -- it picks up where it left off. Download the GGUFs from
Drive whenever convenient (they are already safe).

In [ ]:
!pip install -q unsloth

In [ ]:
# --- Mount Google Drive so outputs survive a runtime disconnect ---
import os
from google.colab import drive
drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/distill_llama_control'
os.makedirs(DRIVE_DIR, exist_ok=True)
print('Artifacts will be saved to:', DRIVE_DIR)
print('Already present:', sorted(os.listdir(DRIVE_DIR)))

In [ ]:
import torch, gc
from unsloth import FastLanguageModel
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

MAX_SEQ = 2048
SEEDS = [42, 123, 7]
dataset = load_dataset('json', data_files='train_chat_llama_final.jsonl', split='train')
print('train examples:', len(dataset))

In [ ]:
import glob, shutil, os

def find_gguf(out):
    # Unsloth may save into '<out>/' or '<out>_gguf/'; search broadly, take newest.
    cands = set(glob.glob(f'{out}/*.gguf')) | set(glob.glob(f'{out}_gguf/*.gguf')) \
          | set(glob.glob(f'{out}*/**/*.gguf', recursive=True))
    cands = sorted(cands, key=os.path.getmtime)
    if not cands:
        raise FileNotFoundError(f'No GGUF produced for {out}. Dirs present: {glob.glob(out + "*")}')
    return cands[-1]

def gguf_in_drive(seed):
    hits = glob.glob(f'{DRIVE_DIR}/qwen3-0.6b-rss-llama-seed{seed}*.gguf')
    return hits[0] if hits else None

def train_seed(seed):
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name='unsloth/Qwen3-0.6B', max_seq_length=MAX_SEQ, load_in_4bit=True, dtype=None)
    model = FastLanguageModel.get_peft_model(
        model, r=32,
        target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
        lora_alpha=32, lora_dropout=0, bias='none',
        use_gradient_checkpointing='unsloth', random_state=seed)

    def fmt(ex):
        return {'text': [tokenizer.apply_chat_template(c, tokenize=False, add_generation_prompt=False, enable_thinking=False) for c in ex['messages']]}
    ds = dataset.map(fmt, batched=True)

    trainer = SFTTrainer(
        model=model, tokenizer=tokenizer, train_dataset=ds,
        args=SFTConfig(dataset_text_field='text', max_seq_length=MAX_SEQ,
            per_device_train_batch_size=2, gradient_accumulation_steps=4,
            warmup_steps=5, num_train_epochs=3, learning_rate=2e-4,
            logging_steps=10, optim='adamw_8bit', weight_decay=0.01,
            lr_scheduler_type='linear', seed=seed, output_dir=f'out_{seed}', report_to='none'))
    trainer = train_on_responses_only(trainer,
        instruction_part='<|im_start|>user\n', response_part='<|im_start|>assistant\n')
    trainer.train()

    # 1) export GGUF locally; robustly locate whatever/wherever Unsloth wrote it
    out = f'qwen3-0.6b-rss-llama-seed{seed}'
    model.save_pretrained_gguf(out, tokenizer, quantization_method='q4_k_m')
    local_gguf = find_gguf(out)

    # 2) PERSIST to Drive immediately -> survives any later disconnect
    dst = f'{DRIVE_DIR}/qwen3-0.6b-rss-llama-seed{seed}.Q4_K_M.gguf'
    shutil.copy(local_gguf, dst)
    model.save_pretrained(f'{DRIVE_DIR}/adapter-seed{seed}')  # small fallback
    print(f'=== seed {seed} PERSISTED -> {dst} ===')

    del model, trainer, ds; gc.collect(); torch.cuda.empty_cache()

for s in SEEDS:
    existing = gguf_in_drive(s)
    if existing:
        print(f'seed {s}: already in Drive ({existing}) -> skip'); continue
    print(f'\n########## TRAINING SEED {s} ##########')
    train_seed(s)

print('\nAll seeds present in Drive:')
for s in SEEDS: print(' ', gguf_in_drive(s))

In [ ]:
# Confirm the durable copies in Drive (safe to download any time)
!ls -lh /content/drive/MyDrive/distill_llama_control/*.gguf

## Bring the 3 control models to Ollama on your Mac

The GGUFs are in Drive at `MyDrive/distill_llama_control/`. Download them (or use
the Drive desktop app / `rclone` to sync) -- there is no download race anymore,
they persist. Map each seed to a parallel `rss-llama-s{1,2,3}` name. For each,
put the GGUF in a folder with a `Modelfile`:

```
FROM ./qwen3-0.6b-rss-llama-seed42.Q4_K_M.gguf
PARAMETER temperature 0
```

Then create all three (seed 42 -> s1, 123 -> s2, 7 -> s3):

```bash
ollama create rss-llama-s1 -f Modelfile.s42
ollama create rss-llama-s2 -f Modelfile.s123
ollama create rss-llama-s3 -f Modelfile.s7
```

(Temperature 0 to match the eval protocol.)

**Alternative to Drive (optional):** push each GGUF to the Hugging Face Hub with
`huggingface_hub.upload_file(...)` using a write token -- a permanent URL,
independent of Drive quota. Drive is the zero-extra-auth default.